### CIFAR-10(Object Detection)(Full Code)
The Dataset Class  
Dataset Transforms  
Normalizing Data  

In [1]:
from torchvision import datasets, transforms

# -------------------------------------------------
# Step 1 : Create Transform
# -------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),                      # Convert image to Tensor (0-1)
    transforms.Normalize(
        (0.5, 0.5, 0.5),                        # Mean
        (0.5, 0.5, 0.5)                         # Standard Deviation
    )
])

# -------------------------------------------------
# Step 2 : Download CIFAR-10 Dataset
# -------------------------------------------------
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# -------------------------------------------------
# Step 3 : Dataset Information
# -------------------------------------------------
print("Training Images :", len(train_dataset))
print("Testing Images  :", len(test_dataset))

# -------------------------------------------------
# Step 4 : Read One Sample
# -------------------------------------------------
image, label = train_dataset[0]

print("\nImage Shape :", image.shape)
print("Label Number:", label)
print("Class Name  :", train_dataset.classes[label])

# -------------------------------------------------
# Step 5 : Pixel Range After Normalization
# -------------------------------------------------
print("\nMinimum Pixel :", image.min())
print("Maximum Pixel :", image.max())

# -------------------------------------------------
# Step 6 : Print All Classes
# -------------------------------------------------
print("\nClasses:")
print(train_dataset.classes)

Training Images : 50000
Testing Images  : 10000

Image Shape : torch.Size([3, 32, 32])
Label Number: 6
Class Name  : frog

Minimum Pixel : tensor(-1.)
Maximum Pixel : tensor(1.)

Classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### Complete Code(all of)

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

# -----------------------------
# Transform
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5,0.5,0.5),
        (0.5,0.5,0.5)
    )
])

# -----------------------------
# Download CIFAR-10
# -----------------------------
train_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# -----------------------------
# Build Bird vs Airplane Dataset
# -----------------------------
bird_airplane = []

for image, label in train_data:
    if label == 0 or label == 2:
        bird_airplane.append((image, 0 if label == 0 else 1))
        # Remap labels:
        # Airplane = 0
        # Bird = 1

# -----------------------------
# Build Fully Connected Model
# -----------------------------
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * 32 * 32, 128),
    nn.ReLU(),
    nn.Linear(128, 2)
)

# -----------------------------
# Loss & Optimizer
# -----------------------------
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

# -----------------------------
# Train on One Sample (Demo)
# -----------------------------
image, label = bird_airplane[0]

image = image.unsqueeze(0)
label = torch.tensor([label])

optimizer.zero_grad()

output = model(image)

loss = loss_fn(output, label)

loss.backward()

optimizer.step()

print("Loss:", loss.item())

# -----------------------------
# Convert to Probabilities
# -----------------------------
probabilities = torch.softmax(output, dim=1)

print("Probabilities:", probabilities)

prediction = torch.argmax(output, dim=1)

classes = ["Airplane", "Bird"]

print("Prediction:", classes[prediction.item()])

# -----------------------------
# Parameter Count
# -----------------------------
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

Loss: 0.6719390153884888
Probabilities: tensor([[0.4893, 0.5107]], grad_fn=<SoftmaxBackward0>)
Prediction: Bird
Total Parameters: 393602
